In [ ]:
!nvidia-smi
!pip install -q ultralytics

from google.colab import drive
drive.mount('/content/drive')   # aquí se guardan el dataset y el modelo para no perderlos

Tue Oct 20 19:54:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Fotos que cuentan como "parque"
POSITIVAS = ["park", "botanical_garden", "formal_garden", "japanese_garden",
             "topiary_garden", "vegetable_garden", "zen_garden", "roof_garden",
             "playground", "picnic_area", "yard"]

# Fotos que cuentan como "no_parque" (calles, interiores, edificios…)
NEGATIVAS = ["street", "alley", "highway", "parking_lot", "parking_garage/outdoor",
             "gas_station", "crosswalk", "downtown", "building_facade",
             "construction_site", "bridge", "bus_station/indoor", "office",
             "living_room", "kitchen", "bedroom", "shopping_mall/indoor",
             "supermarket", "restaurant", "garage/indoor"]

MAX_POS = 2000   # imágenes por categoría positiva  (~22k en total)
MAX_NEG = 1100   # imágenes por categoría negativa  (~22k en total, queda balanceado)

# Comprobar que los nombres existen en Places365
!wget -q http://data.csail.mit.edu/places/places365/filelist_places365-standard.tar
!tar -xf filelist_places365-standard.tar
cats = {l.split()[0][3:] for l in open('categories_places365.txt')}
faltan = [c for c in POSITIVAS + NEGATIVAS if c not in cats]
print("Faltan:", faltan if faltan else "ninguna, todo bien")


Faltan: ninguna, todo bien


In [ ]:
import tarfile, urllib.request, re
from pathlib import Path

URL = "http://data.csail.mit.edu/places/places365/train_256_places365standard.tar"
RAW = Path('/content/raw')
pos_set = set(POSITIVAS)
cuenta = {c: 0 for c in POSITIVAS + NEGATIVAS}
limite = lambda c: MAX_POS if c in pos_set else MAX_NEG
patron = re.compile(r'/[a-z]/(.+)/[^/]+\.jpg$')

with urllib.request.urlopen(URL) as resp, tarfile.open(fileobj=resp, mode='r|') as tar:
    for i, m in enumerate(tar):
        if i % 200_000 == 0:
            print(f"{i:,} archivos revisados… actual: {m.name}")
        if not m.isfile():
            continue
        r = patron.search('/' + m.name)
        if not r or r.group(1) not in cuenta:
            continue
        cat = r.group(1)
        if cuenta[cat] >= limite(cat):
            continue
        clase = 'parque' if cat in pos_set else 'no_parque'
        destino = RAW / clase / f"{cat.replace('/', '_')}_{cuenta[cat]:05d}.jpg"
        destino.parent.mkdir(parents=True, exist_ok=True)
        with tar.extractfile(m) as f:
            destino.write_bytes(f.read())
        cuenta[cat] += 1
        if all(cuenta[c] >= limite(c) for c in cuenta):
            print("Todas las categorías completas")
            break

print(cuenta)


0 archivos revisados… actual: data_256/a/airfield/00000001.jpg
200,000 archivos revisados… actual: data_256/b/barn/00002333.jpg
400,000 archivos revisados… actual: data_256/c/candy_store/00004762.jpg
600,000 archivos revisados… actual: data_256/d/dining_room/00004368.jpg
800,000 archivos revisados… actual: data_256/g/glacier/00000881.jpg
1,000,000 archivos revisados… actual: data_256/k/kitchen/00003816.jpg
1,200,000 archivos revisados… actual: data_256/o/ocean/00004550.jpg
1,400,000 archivos revisados… actual: data_256/r/residential_neighborhood/00004736.jpg
1,600,000 archivos revisados… actual: data_256/s/swimming_hole/00000716.jpg
1,800,000 archivos revisados… actual: data_256/z/zen_garden/00001541.jpg
Todas las categorías completas
{'park': 2000, 'botanical_garden': 2000, 'formal_garden': 2000, 'japanese_garden': 2000, 'topiary_garden': 2000, 'vegetable_garden': 2000, 'zen_garden': 2000, 'roof_garden': 2000, 'playground': 2000, 'picnic_area': 2000, 'yard': 2000, 'street': 1100, 'all

In [ ]:
import random, shutil
DATA = Path('/content/dataset')
random.seed(42)

for clase in ['parque', 'no_parque']:
    imgs = sorted((RAW / clase).glob('*.jpg'))
    random.shuffle(imgs)
    n_val = int(len(imgs) * 0.15)
    for split, lote in [('val', imgs[:n_val]), ('train', imgs[n_val:])]:
        d = DATA / split / clase
        d.mkdir(parents=True, exist_ok=True)
        for p in lote:
            shutil.move(str(p), d / p.name)
    print(clase, len(imgs), "imágenes")

!cd /content && zip -qr /content/drive/MyDrive/georeporte_parques_dataset.zip dataset


parque 22000 imágenes
no_parque 22000 imágenes


In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n-cls.pt')   # modelo de clasificación pequeño y rápido
model.train(
    data='/content/dataset',
    epochs=20,
    imgsz=224,
    batch=128,
    patience=5,                  # se detiene si deja de mejorar
    project='/content/drive/MyDrive/georeporte_parques',
    name='v1',
    exist_ok=True,
)


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, 

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7db174591480>
curves: []
curves_results: []
fitness: 0.9646969735622406
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.9293939471244812, 'metrics/accuracy_top5': 1.0, 'fitness': 0.9646969735622406}
save_dir: PosixPath('/content/drive/MyDrive/georeporte_parques/v1')
speed: {'preprocess': 0.18615709045455087, 'inference': 0.2760388821212528, 'loss': 5.597636393786436e-05, 'postprocess': 9.669348490647144e-05}
top1: 0.9293939471244812
top5: 1.0

In [ ]:
best = YOLO('/content/drive/MyDrive/georeporte_parques/v1/weights/best.pt')
metrics = best.val(data='/content/dataset')
print("Precisión top-1:", metrics.top1)

best.export(format='onnx', imgsz=224)
# El archivo queda en: /content/drive/MyDrive/georeporte_parques/v1/weights/best.onnx


Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n-cls summary (fused): 47 layers, 1,528,586 parameters, 0 gradients, 3.2 GFLOPs
train: /content/dataset/train... found 37400 images in 2 classes ✅ 
val: /content/dataset/val... found 6600 images in 2 classes ✅ 
test: None...
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 731.5±362.6 MB/s, size: 12.5 KB)
val: Scanning /content/dataset/val... 6600 images, 0 corrupt: 100% ━━━━━━━━━━━━ 6600/6600 922.7Mit/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 413/413 22.4it/s 18.4s
                   all       0.93          1
Speed: 0.1ms preprocess, 1.1ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/runs/classify/val
Precisión top-1: 0.9295454621315002
Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.

'/content/drive/MyDrive/georeporte_parques/v1/weights/best.onnx'

In [ ]:
from google.colab import files
from PIL import Image

subidas = files.upload()
modelo = YOLO('/content/drive/MyDrive/georeporte_parques/v1/weights/best.onnx', task='classify')

for nombre in subidas:
    img = Image.open(nombre).convert('RGB')
    r = modelo.predict(img, imgsz=224, verbose=False)[0]
    print(f"{nombre}: {r.names[r.probs.top1]} ({float(r.probs.top1conf):.1%})")

Saving bache 2.jfif to bache 2.jfif
Loading /content/drive/MyDrive/georeporte_parques/v1/weights/best.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.31.0 with CPUExecutionProvider
WARNING ⚠️ CUDA requested but CUDAExecutionProvider not available. Using CPU...
bache 2.jfif: no_parque (77.0%)


In [ ]:
from PIL import Image

for nombre in subidas:
    img = Image.open(nombre).convert('RGB')
    r = modelo.predict(img, imgsz=224, verbose=False)[0]
    print(f"{nombre}: {r.names[r.probs.top1]} ({float(r.probs.top1conf):.1%})")

parque basura.jfif: parque (98.1%)
parque.jfif: parque (99.8%)


In [ ]:
import os, shutil, random
from pathlib import Path

DATA = Path('/content/dataset')

# Si Colab se reinició, recuperar el dataset desde Drive
if not DATA.exists():
    os.system('unzip -q /content/drive/MyDrive/georeporte_parques_dataset.zip -d /content')

# Borrar las fotos de zen_garden (en train y en val)
borradas = 0
for p in DATA.glob('*/parque/zen_garden_*.jpg'):
    p.unlink()
    borradas += 1
print("Fotos de zen_garden borradas:", borradas)

# Borrar el caché viejo de YOLO para que relea las carpetas
for c in DATA.glob('*.cache'):
    c.unlink()

Fotos de zen_garden borradas: 2000


In [ ]:
import zipfile

EXT = Path('/content/externos')
EXT.mkdir(exist_ok=True)

HF = "https://huggingface.co/datasets/keremberke/pothole-segmentation/resolve/main/data"
fuentes = {
    'bache':  [f"{HF}/train.zip", f"{HF}/valid.zip", f"{HF}/test.zip"],
    'grieta': ["https://github.com/ultralytics/assets/releases/download/v0.0.0/crack-seg.zip"],
}

for nombre, urls in fuentes.items():
    for i, url in enumerate(urls):
        zip_path = EXT / f"{nombre}_{i}.zip"
        if not zip_path.exists() or zip_path.stat().st_size == 0:
            os.system(f'wget -q -O {zip_path} "{url}"')
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(EXT / nombre / str(i))
        print(f"Descomprimido {zip_path.name}")

random.seed(42)
for nombre in fuentes:
    imgs = sorted(p for p in (EXT / nombre).rglob('*')
                  if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
    if not imgs:
        print(f"⚠️ No se encontraron imágenes de {nombre}")
        continue
    random.shuffle(imgs)
    n_val = int(len(imgs) * 0.15)
    for split, lote in [('val', imgs[:n_val]), ('train', imgs[n_val:])]:
        d = DATA / split / 'no_parque'
        for j, p in enumerate(lote):
            shutil.copy(p, d / f"{nombre}_{split}_{j:05d}{p.suffix.lower()}")
    print(nombre, len(imgs), "imágenes agregadas")

Descomprimido bache_0.zip
Descomprimido bache_1.zip
Descomprimido bache_2.zip
Descomprimido grieta_0.zip
bache 90 imágenes agregadas
grieta 4029 imágenes agregadas


In [ ]:
!ls -la /content/externos
!wget -O /tmp/prueba_grieta.zip "https://github.com/ultralytics/assets/releases/download/v0.0.0/crack-seg.zip" 2>&1 | tail -5
!wget -O /tmp/prueba_bache.zip "https://huggingface.co/datasets/keremberke/pothole-segmentation/resolve/main/data/train.zip" 2>&1 | tail -5
!ls -la /tmp/prueba_*.zip

total 100628
drwxr-xr-x 2 root root     4096 Oct 20 21:03 .
drwxr-xr-x 1 root root     4096 Oct 20 21:03 ..
-rw-r--r-- 1 root root  6223723 Oct 20 21:03 bache_0.zip
-rw-r--r-- 1 root root   328798 Oct 20 21:03 bache_1.zip
-rw-r--r-- 1 root root   382846 Oct 20 21:03 bache_2.zip
-rw-r--r-- 1 root root 96089865 Aug 21  2025 grieta_0.zip
 93750K .......... .......... .......... .......... .......... 99% 47.2M 0s
 93800K .......... .......... .......... .......              100% 54.1M=0.6s

2026-10-20 21:04:52 (144 MB/s) - ‘/tmp/prueba_grieta.zip’ saved [96089865/96089865]

  6000K .......... .......... .......... .......... .......... 99%  326M 0s
  6050K .......... .......... .......                         100%  270M=0.5s

2026-10-20 21:04:53 (11.6 MB/s) - ‘/tmp/prueba_bache.zip’ saved [6223723/6223723]

-rw-r--r-- 1 root root  6223723 Oct 20 21:04 /tmp/prueba_bache.zip
-rw-r--r-- 1 root root 96089865 Aug 21  2025 /tmp/prueba_grieta.zip


In [ ]:
for split in ['train', 'val']:
    for clase in ['parque', 'no_parque']:
        print(split, clase, len(list((DATA / split / clase).iterdir())))

!cd /content && zip -qr /content/drive/MyDrive/georeporte_parques_dataset_v2.zip dataset

train parque 16954
train no_parque 22202
val parque 3046
val no_parque 3917


In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n-cls.pt')
model.train(
    data='/content/dataset',
    epochs=20,
    imgsz=224,
    batch=128,
    patience=5,
    project='/content/drive/MyDrive/georeporte_parques',
    name='v2',
    exist_ok=True,
)

Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=v2, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto, ove

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7db0b9aea120>
curves: []
curves_results: []
fitness: 0.9794628620147705
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.958925724029541, 'metrics/accuracy_top5': 1.0, 'fitness': 0.9794628620147705}
save_dir: PosixPath('/content/drive/MyDrive/georeporte_parques/v2')
speed: {'preprocess': 0.14058425233485802, 'inference': 0.2137201140313119, 'loss': 5.049202850787136e-05, 'postprocess': 9.283943778562581e-05}
top1: 0.958925724029541
top5: 1.0

In [ ]:
from google.colab import files
from PIL import Image

v2 = YOLO('/content/drive/MyDrive/georeporte_parques/v2/weights/best.pt')
v2.export(format='onnx', imgsz=224)

modelo = YOLO('/content/drive/MyDrive/georeporte_parques/v2/weights/best.onnx', task='classify')
subidas = files.upload()
for nombre in subidas:
    img = Image.open(nombre).convert('RGB')
    r = modelo.predict(img, imgsz=224, verbose=False)[0]
    print(f"{nombre}: {r.names[r.probs.top1]} ({float(r.probs.top1conf):.1%})")

Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
YOLO11n-cls summary (fused): 47 layers, 1,528,586 parameters, 0 gradients, 0.4 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/georeporte_parques/v2/weights/best.pt' with input shape (1, 3, 224, 224) BCHW and output shape(s) (1, 2) (3.0 MB)

ONNX: starting export with onnx 1.23.2 opset 18...
ONNX: slimming with onnxslim 0.1.98...
ONNX: export success ✅ 0.5s, saved as '/content/drive/MyDrive/georeporte_parques/v2/weights/best.onnx' (5.9 MB)

Export complete (0.6s)
Results saved to /content/drive/MyDrive/georeporte_parques/v2/weights/best.onnx
Predict:         yolo predict task=classify model=/content/drive/MyDrive/georeporte_parques/v2/weights/best.onnx imgsz=224 
Validate:        yolo val task=classify model=/content/drive/MyDrive/georeporte_parques/v2/weights/best.onnx imgsz=224 data=/content/dataset 
Visualize:       https://netron.app


Saving ARBOL-RAMAS-CABLES.jpg to ARBOL-RAMAS-CABLES.jpg
Saving images (1).jfif to images (1) (1).jfif
Saving images (2).jfif to images (2) (1).jfif
Saving images (4).jfif to images (4) (1).jfif
Saving images.jfif to images (20).jfif
Loading /content/drive/MyDrive/georeporte_parques/v2/weights/best.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.31.0 with CPUExecutionProvider
WARNING ⚠️ CUDA requested but CUDAExecutionProvider not available. Using CPU...
ARBOL-RAMAS-CABLES.jpg: parque (75.1%)
images (1) (1).jfif: parque (99.6%)
images (2) (1).jfif: parque (97.5%)
images (4) (1).jfif: parque (99.2%)
images (20).jfif: parque (98.7%)
